# CylinderFlow assignment: deterministic prediction vs conditional flow matching

In this assignment you will model the future velocity field of a two-dimensional flow around a cylinder on an **unstructured, nonuniform triangular mesh**.

You will build two predictors using the **same supplied data**:

1. a **deterministic predictor** trained as a point predictor; and
2. a **conditional flow-matching predictor** that represents a distribution of possible future velocity fields.

The neural-network architecture is **not prescribed**. Your model should be able to use the information supplied by the mesh and the current flow state.

This notebook provides the problem specification, data loading, dataset class, data inspection, and the final evaluation code. Model architecture, optimization, training, and the numerical details of flow-matching sampling are your task.


## 1. What is in the dataset?

The cylinder-flow dataset is based on the MeshGraphNets benchmark introduced by Pfaff et al. in [*Learning Mesh-Based Simulation with Graph Networks*](https://arxiv.org/abs/2010.03409) (ICLR 2021). The data consist of multiple independent **trajectories** of two-dimensional flow around a cylinder.

The assignment file is intended to contain:

- **500 training trajectories**;
- **20 validation trajectories**;
- **20 held-out test trajectories**.

Each trajectory contains:

- **600 time states** separated by $\Delta t=0.01\,\mathrm{s}$, corresponding to times $0,0.01,\ldots,5.99\,\mathrm{s}$;
- one **static 2D triangular mesh** that is reused for all 600 states in that trajectory;
- a 2D velocity vector $(u_x,u_y)$ at every mesh node and every time state;
- a node-type label describing whether a node is an interior node, inflow node, outflow node, or wall boundary node.

### Important mesh properties

The mesh is **unstructured and spatially nonuniform**. It is not a Cartesian image grid, so the velocity field should not be reshaped into a rectangular image.

The number of nodes is **not identical across trajectories**. In the supplied training data, trajectories have roughly 600 nodes on average (the current training set spans approximately 552--898 nodes). Triangle connectivity also varies across trajectories.

Within a single trajectory, however, the mesh is fixed in time: only the velocity field changes.

The loading cells below verify the exact split sizes and mesh statistics stored in your copy of the data file.


### Array structure for one trajectory

For a trajectory with \(T\) time states, \(N\) mesh nodes, and \(M\) triangles:

| key | shape | meaning |
|---|---:|---|
| `mesh_pos` | `(N, 2)` | node coordinates $(x,y)$ |
| `cells` | `(M, 3)` | the three node indices of every triangular cell |
| `node_type` | `(N,)` | integer boundary/interior label for each node |
| `velocity` | `(T, N, 2)` | velocity components $(u_x,u_y)$ |

The node labels used in this assignment are:

- `0`: interior / normal node
- `4`: inflow node
- `5`: outflow node
- `6`: wall boundary node

The **dynamic nodes** for the prediction loss are interior and outflow nodes (`0` and `5`).

In the supplied dataset, inflow (`4`) and wall (`6`) velocities are prescribed boundary values and are constant in time. They therefore do not need to be learned as evolving state variables. During autoregressive rollout, restore these known boundary values after every prediction step.


In [ ]:
%pip -q install h5py tqdm

In [ ]:
from pathlib import Path
import math
import random

import h5py
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Download and load the supplied data file

The code below will download and place the supplied HDF5 file next to this notebook.

The HDF5 file is organized as

```text
/train/traj_0000/...
/train/traj_0001/...
/valid/traj_0000/...
/test/traj_0000/...
```

Each trajectory group contains the four arrays described above.

In [ ]:
# ============================================================
# Download dataset
# ============================================================
DATA_PATH = Path("cylinderflow_assignment.h5")

if not DATA_PATH.exists():
    !pip -q install gdown

    import gdown

    DATA_URL = (
        "https://drive.google.com/file/d/1Gqz9b4hfoDaOYvupbHlf1pl_48AP-oZ4/view?usp=sharing"
    )

    print("Downloading dataset...")

    gdown.download(
        url=DATA_URL,
        output=str(DATA_PATH),
        quiet=False,
    )

    print(
        "Download complete:",
        DATA_PATH,
    )

else:
    print(
        "Dataset already available:",
        DATA_PATH,
    )

In [ ]:
DATA_PATH = Path("cylinderflow_assignment.h5")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Change DATA_PATH to the supplied dataset file."
    )

with h5py.File(DATA_PATH, "r") as f:
    DT = float(f.attrs["dt"])
    HORIZON_STEPS = int(f.attrs["prediction_horizon_steps"])
    PREDICTION_DT = float(f.attrs["prediction_horizon_seconds"])

    print("Dataset:", f.attrs.get("dataset_name", "CylinderFlow"))
    print(f"Stored time step: {DT:.3f} s")
    print(f"Prediction horizon: {HORIZON_STEPS} states = {PREDICTION_DT:.1f} s")

    for split in ["train", "valid", "test"]:
        ids = sorted(f[split].keys())
        node_counts = np.array([f[split][tid]["mesh_pos"].shape[0] for tid in ids])
        tri_counts = np.array([f[split][tid]["cells"].shape[0] for tid in ids])
        n_states = np.array([f[split][tid]["velocity"].shape[0] for tid in ids])

        print(f"\n{split.upper()}")
        print("  trajectories:", len(ids))
        print("  states/trajectory:", sorted(set(n_states.tolist())))
        print(
            "  nodes: min / mean / max =",
            int(node_counts.min()),
            f"{node_counts.mean():.1f}",
            int(node_counts.max()),
        )

### Trajectories versus training pairs

The learning task uses a **direct 1-second prediction horizon**. Since the stored time spacing is $0.01\,\mathrm{s}$, the target for state index $t$ is state index $t+100$.

Let $x_t$ denote the **full flow state at physical time $t$ seconds**. The training pairs are sliding 1-second pairs such as

$
(x_{0.00},x_{1.00}),\quad
(x_{0.01},x_{1.01}),\quad
(x_{0.02},x_{1.02}),\quad \ldots
$

and later in the same trajectory,

$
(x_{1.00},x_{2.00}),\quad
(x_{1.05},x_{2.05}),\quad \ldots,\quad
(x_{4.99},x_{5.99}).
$

The important point is:

> **1 second is the separation between the input and target, not the spacing between consecutive training examples.**

A trajectory with 600 stored states provides

$
600-100=500
$

possible 1-second input/target pairs. Thus 500 training trajectories correspond to $250{,}000$ possible pairs, but they still come from only 500 independent trajectories.


## 3. Dataset class

The class below returns raw physical quantities and makes no assumption about your model architecture.

Because different trajectories can have different numbers of nodes, a conventional tensor batch cannot in general stack examples from different meshes. The provided `variable_mesh_collate` therefore returns a Python list when `batch_size > 1`. You may process samples individually, group compatible samples yourself, or implement another batching strategy.


In [ ]:
NORMAL = 0
INFLOW = 4
OUTFLOW = 5
WALL_BOUNDARY = 6


def dynamic_node_mask(node_type):
    """Nodes whose velocity is predicted/evaluated dynamically."""
    return (node_type == NORMAL) | (node_type == OUTFLOW)


class CylinderFlowPairDataset(Dataset):
    """
    Direct-horizon input/target pairs from a variable-size triangular-mesh dataset.

    One item contains the static mesh plus
        velocity_t       = u(t)
        velocity_target  = u(t + 1 s)
        delta_velocity   = velocity_target - velocity_t
    """

    def __init__(self, h5_path, split="train"):
        self.h5_path = str(h5_path)
        self.split = split
        self._h5 = None
        self._static_cache = {}

        with h5py.File(self.h5_path, "r") as f:
            if split not in f:
                raise KeyError(f"Unknown split {split!r}")

            self.dt = float(f.attrs["dt"])
            self.horizon_steps = int(f.attrs["prediction_horizon_steps"])
            self.prediction_dt = float(f.attrs["prediction_horizon_seconds"])
            self.traj_ids = sorted(f[split].keys())

            self.index = []
            for traj_id in self.traj_ids:
                T = int(f[split][traj_id]["velocity"].shape[0])
                n_valid = T - self.horizon_steps
                self.index.extend((traj_id, t) for t in range(n_valid))

    def _file(self):
        # Each DataLoader worker gets its own lazily opened handle.
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_h5"] = None
        state["_static_cache"] = {}
        return state

    def __len__(self):
        return len(self.index)

    def _static(self, traj_id):
        if traj_id not in self._static_cache:
            g = self._file()[self.split][traj_id]
            self._static_cache[traj_id] = {
                "mesh_pos": torch.from_numpy(g["mesh_pos"][:]).float(),
                "cells": torch.from_numpy(g["cells"][:].astype(np.int64)).long(),
                "node_type": torch.from_numpy(g["node_type"][:].astype(np.int64)).long(),
            }
        return self._static_cache[traj_id]

    def __getitem__(self, idx):
        traj_id, t = self.index[idx]
        g = self._file()[self.split][traj_id]
        static = self._static(traj_id)

        u0 = torch.from_numpy(g["velocity"][t]).float()
        u1 = torch.from_numpy(g["velocity"][t + self.horizon_steps]).float()

        return {
            "trajectory_id": traj_id,
            "time_index": t,
            "time_s": t * self.dt,
            "target_time_index": t + self.horizon_steps,
            "target_time_s": (t + self.horizon_steps) * self.dt,
            "mesh_pos": static["mesh_pos"],
            "cells": static["cells"],
            "node_type": static["node_type"],
            "dynamic_mask": dynamic_node_mask(static["node_type"]),
            "velocity_t": u0,
            "velocity_target": u1,
            "delta_velocity": u1 - u0,
        }

    def close(self):
        if self._h5 is not None:
            self._h5.close()
            self._h5 = None


def variable_mesh_collate(batch):
    # Mesh sizes vary, so do not torch.stack different trajectories automatically.
    return batch


train_dataset = CylinderFlowPairDataset(DATA_PATH, "train")
valid_dataset = CylinderFlowPairDataset(DATA_PATH, "valid")
test_dataset = CylinderFlowPairDataset(DATA_PATH, "test")

print("Training pairs:", len(train_dataset))
print("Validation pairs:", len(valid_dataset))
print("Test pairs:", len(test_dataset))

### Concrete examples of the 1-second pairs

The following cell prints several pairs from the first training trajectory. It is included to make the sliding horizon explicit. For example, a pair beginning at $1.05\,\mathrm{s}$ has its target at $2.05\,\mathrm{s}$, not at the next stored state $1.06\,\mathrm{s}$.


In [ ]:
# Show several explicit pairs from the first training trajectory.
first_traj_id = train_dataset.traj_ids[0]

example_start_steps = [
    0,      # 0.00 -> 1.00 s
    1,      # 0.01 -> 1.01 s
    100,    # 1.00 -> 2.00 s
    105,    # 1.05 -> 2.05 s
    499,    # 4.99 -> 5.99 s
]

print("Examples from", first_traj_id)

for t_idx in example_start_steps:
    dataset_idx = train_dataset.index.index((first_traj_id, t_idx))
    item = train_dataset[dataset_idx]

    print(
        f"x_{item['time_s']:.2f}  ->  "
        f"x_{item['target_time_s']:.2f}  "
        f"(indices {item['time_index']} -> {item['target_time_index']})"
    )


In [ ]:
# Inspect one direct 1-second pair.
example = train_dataset[0]

for key in [
    "mesh_pos",
    "cells",
    "node_type",
    "velocity_t",
    "velocity_target",
    "delta_velocity",
]:
    print(f"{key:18s}", tuple(example[key].shape))

print("input time:", example["time_s"], "s")
print("target time:", example["target_time_s"], "s")
print("dynamic nodes:", int(example["dynamic_mask"].sum()), "/", len(example["node_type"]))


## 4. Visualize the mesh and one 1-second pair

The left panel shows the triangular connectivity. The other panels show velocity magnitude
$
|\mathbf u|=\sqrt{u_x^2+u_y^2}
$
at the input and target times.

Notice that the mesh spacing is nonuniform and the geometry is represented directly by node coordinates and triangle connectivity.


In [ ]:
pos = example["mesh_pos"].numpy()
cells = example["cells"].numpy()

speed0 = torch.linalg.vector_norm(example["velocity_t"], dim=-1).numpy()
speed1 = torch.linalg.vector_norm(example["velocity_target"], dim=-1).numpy()

vmin = min(float(speed0.min()), float(speed1.min()))
vmax = max(float(speed0.max()), float(speed1.max()))

fig, axes = plt.subplots(1, 3, figsize=(17, 4), constrained_layout=True)

axes[0].triplot(pos[:, 0], pos[:, 1], cells, linewidth=0.35)
axes[0].set_title(f"Triangular mesh ({len(pos)} nodes)")

m0 = axes[1].tripcolor(
    pos[:, 0], pos[:, 1], cells, speed0,
    shading="gouraud", vmin=vmin, vmax=vmax,
)
axes[1].set_title(f"Input speed, t={example['time_s']:.2f} s")

m1 = axes[2].tripcolor(
    pos[:, 0], pos[:, 1], cells, speed1,
    shading="gouraud", vmin=vmin, vmax=vmax,
)
axes[2].set_title(f"Target speed, t={example['target_time_s']:.2f} s")

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

fig.colorbar(m1, ax=axes[1:], label="Velocity magnitude")
plt.show()

## 5. Prediction task

For every training example you are given the current state
$
\mathcal C_t=
\{\mathbf u_t,\;\text{mesh positions},\;\text{triangle connectivity},\;\text{node types}\}
$
and the target state one physical second later,
$
\mathbf u_{t+1\mathrm{s}}.
$
It is convenient to model the 1-second velocity increment
$
\Delta\mathbf u_t=
\mathbf u_{t+1\mathrm{s}}-\mathbf u_t.
$

### Deterministic model

You should construct a deterministic model that predicts one increment (or equivalently one future state) from the conditioning state. A standard deterministic baseline minimizes mean-squared error on the dynamic nodes.

### Conditional flow matching

You should construct a conditional flow-matching model for the distribution
$
p(\Delta\mathbf u_t\mid\mathcal C_t).
$
For conditional flow matching, a common training construction is:
$
\epsilon\sim\mathcal N(0,I),
\qquad
\tau\sim U[0,1],
$

$
z_\tau=(1-\tau)\epsilon+\tau\widetilde{\Delta\mathbf u}_t,
$

with target vector field

$
v^*=
\widetilde{\Delta\mathbf u}_t-\epsilon.
$

A sample is generated by integrating the learned ODE from $\tau=0$ to $1$, starting from fresh Gaussian noise.

Only **dynamic nodes** (`NORMAL` and `OUTFLOW`) should contribute to the prediction loss. During rollout, known inflow and wall values should remain fixed.

### What is not prescribed

You choose the model architecture, data normalization, feature representation, optimizer, learning-rate schedule, and the flow-matching sampler. Your design must support variable-size, nonuniform meshes.


## 6. Your model and training code

Implement your models and training procedure below.

Use only the **training split** for fitting model parameters and training statistics. Use the **validation split** for model selection. Do not tune on the test split.

For the final evaluation code later in this notebook, your implementation only needs to provide two small adapter functions that produce a **full next velocity field** after one 1-second model step. These adapter functions must return velocities in physical, unnormalized units.


In [ ]:
# ============================================================
# TODO: your deterministic model, flow-matching model, training,
# validation, checkpointing, and inference utilities go here.
# ============================================================

# 7. Final evaluation: wake temporal activity

The supplied evaluation code computes how strongly the velocity field changes over a finite time interval in a fixed downstream wake region. The following sections explain the metric and protocol implemented below. **Your task is to connect your trained models through the two adapter functions, run the provided evaluation and plotting functions, and interpret the results alongside rollout visualizations**.

For two states at times $t_0$ and $t_1$, define at each mesh node $i$,
$
a_i(t_0,t_1)=
\frac{
\left\|
\mathbf u_i(t_1)-\mathbf u_i(t_0)
\right\|_2
}{
t_1-t_0
}.
$

This uses the full velocity vector, so changes in both velocity magnitude and direction contribute.

For a triangle $K$ with nodes $i,j,k$, the nodal scalar $a$ is linearly interpolated over the triangle. Its triangle-average value is therefore
$
a_K=
\frac{a_i+a_j+a_k}{3}.
$

For all triangles whose centroids lie inside the fixed wake region, define the area-weighted temporal activity
$
D_{\mathrm{wake}}(t_0,t_1)=
\frac{
\sum_{K\in\mathrm{wake}} A_K\,a_K
}{
\sum_{K\in\mathrm{wake}} A_K
}.
$

### Interpretation

A larger value means that the wake velocity field changes more strongly between the two endpoint states. A smaller value means less finite-time change.

The first rollout interval $0\rightarrow1\,\mathrm{s}$ is treated as a burn-in interval and is not used to calculate the metric. The reported metric begins with the interval $1\rightarrow2\,\mathrm{s}$.


## 8. Wake region used by the metric

The provided metric function uses a fixed physical rectangle downstream of the cylinder:

$
0.55\le x\le1.55,
\qquad
0.03\le y\le0.39.
$

A triangle contributes if its centroid lies inside this rectangle. The same region is used for ground truth, the deterministic model, and every generative sample.

The area weighting is important because the mesh is nonuniform: a large triangle represents more physical area than a small triangle.

The region is fixed in advance and must not be tuned separately for different models.


In [ ]:
# ============================================================
# Fixed wake region and absolute temporal-activity metric
# ============================================================

METRIC_WAKE_X_MIN = 0.55
METRIC_WAKE_X_MAX = 1.55
METRIC_WAKE_Y_MIN = 0.03
METRIC_WAKE_Y_MAX = 0.39


def triangle_areas_and_centroids(mesh_pos, cells):
    if torch.is_tensor(mesh_pos):
        mesh_pos = mesh_pos.detach().cpu().numpy()

    if torch.is_tensor(cells):
        cells = cells.detach().cpu().numpy()

    pos = np.asarray(mesh_pos, dtype=np.float64)
    cells = np.asarray(cells, dtype=np.int64)

    p0 = pos[cells[:, 0]]
    p1 = pos[cells[:, 1]]
    p2 = pos[cells[:, 2]]

    cross = (
        (p1[:, 0] - p0[:, 0])
        *
        (p2[:, 1] - p0[:, 1])
        -
        (p1[:, 1] - p0[:, 1])
        *
        (p2[:, 0] - p0[:, 0])
    )

    areas = 0.5 * np.abs(cross)
    centroids = (p0 + p1 + p2) / 3.0

    return areas, centroids


def wake_triangle_mask(centroids):
    x = centroids[:, 0]
    y = centroids[:, 1]

    return (
        (x >= METRIC_WAKE_X_MIN)
        &
        (x <= METRIC_WAKE_X_MAX)
        &
        (y >= METRIC_WAKE_Y_MIN)
        &
        (y <= METRIC_WAKE_Y_MAX)
    )


def wake_velocity_temporal_activity(
    mesh_pos,
    cells,
    velocity_start,
    velocity_end,
    delta_t,
):
    """
    Area-weighted wake average of

        ||u(t1) - u(t0)||_2 / (t1 - t0)

    using the full velocity vector.
    """

    if delta_t <= 0:
        raise ValueError("delta_t must be positive.")

    if torch.is_tensor(velocity_start):
        velocity_start = velocity_start.detach().cpu().numpy()

    if torch.is_tensor(velocity_end):
        velocity_end = velocity_end.detach().cpu().numpy()

    cells_np = (
        cells.detach().cpu().numpy()
        if torch.is_tensor(cells)
        else np.asarray(cells)
    ).astype(np.int64)

    v0 = np.asarray(velocity_start, dtype=np.float64)
    v1 = np.asarray(velocity_end, dtype=np.float64)

    # Full-vector finite-time change rate at each node.
    node_activity = (
        np.linalg.norm(
            v1 - v0,
            axis=-1,
        )
        /
        float(delta_t)
    )

    areas, centroids = triangle_areas_and_centroids(
        mesh_pos,
        cells_np,
    )

    wake = wake_triangle_mask(centroids)

    if not np.any(wake):
        raise RuntimeError(
            "No mesh triangles fall inside the fixed wake region."
        )

    # Exact triangle mean of the linearly interpolated nodal scalar.
    tri_activity = (
        node_activity[cells_np[:, 0]]
        +
        node_activity[cells_np[:, 1]]
        +
        node_activity[cells_np[:, 2]]
    ) / 3.0

    return float(
        np.sum(
            areas[wake]
            *
            tri_activity[wake]
        )
        /
        np.sum(areas[wake])
    )


In [ ]:
# Visualize the fixed evaluation region on the example mesh.
from matplotlib.patches import Rectangle

example_pos = example["mesh_pos"].numpy()
example_cells = example["cells"].numpy()

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.triplot(example_pos[:, 0], example_pos[:, 1], example_cells, linewidth=0.3)

rect = Rectangle(
    (METRIC_WAKE_X_MIN, METRIC_WAKE_Y_MIN),
    METRIC_WAKE_X_MAX - METRIC_WAKE_X_MIN,
    METRIC_WAKE_Y_MAX - METRIC_WAKE_Y_MIN,
    fill=False,
    linewidth=2,
)
ax.add_patch(rect)
ax.set_title("Fixed wake region used for the temporal-activity metric")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal")
plt.show()

## 10. Rollout evaluation protocol

The provided evaluation code implements the following protocol for the final comparison. Run it after implementing the two model adapter functions below:

- The evaluator uses **20 held-out test trajectories**;
- Every rollout starts from the same ground-truth state at $t=0$;
- Each model step advances the model by **1 second**;
- The rollout code generates model states at nominal times $t=0,1,2,3,4,5,6\,\mathrm{s}$;
- The evaluator automatically excludes $0\rightarrow1\,\mathrm{s}$ from the metric as burn-in; students do not need to implement this exclusion;
- The evaluator calculates the metric on the intervals
  $
  1\rightarrow2,\quad
  2\rightarrow3,\quad
  3\rightarrow4,\quad
  4\rightarrow5,\quad
  5\rightarrow6\,\mathrm{s};
  $
- The evaluator runs **5 flow-matching rollouts per test trajectory** with distinct random seeds by repeatedly calling your `generative_next_velocity` adapter. You must implement the conditional sampling procedure, including noise generation using the supplied random generator and numerical integration of the learned flow-matching ODE;
- The evaluator computes the absolute temporal-activity value separately for every generative rollout;
- The supplied aggregation function averages the 5 generative activity values within each trajectory/interval first, then summarizes across trajectories.

### Final ground-truth endpoint

The stored trajectory ends at $5.99\,\mathrm{s}$, not exactly $6.00\,\mathrm{s}$. Therefore, for the final displayed interval:

- deterministic and flow-matching rollouts use their nominal $5.00\rightarrow6.00\,\mathrm{s}$ states and divide by $1.00\,\mathrm{s}$;
- ground truth uses the available $5.00\rightarrow5.99\,\mathrm{s}$ states and divides by the actual $0.99\,\mathrm{s}$.

The final GT point is therefore a $0.99\,\mathrm{s}$ reference for the nominal 6-second endpoint.

This metric measures **finite-time temporal activity**, not pointwise prediction accuracy and not the spatial location or phase of individual vortices. Use it together with rollout visualizations.


In [ ]:
def load_full_trajectory(h5_path, split, traj_id):
    """Load one full trajectory into CPU torch tensors."""
    with h5py.File(h5_path, "r") as f:
        g = f[split][traj_id]
        return {
            "trajectory_id": traj_id,
            "mesh_pos": torch.from_numpy(g["mesh_pos"][:]).float(),
            "cells": torch.from_numpy(g["cells"][:].astype(np.int64)).long(),
            "node_type": torch.from_numpy(g["node_type"][:].astype(np.int64)).long(),
            "velocity": torch.from_numpy(g["velocity"][:]).float(),
        }


def test_trajectory_ids(h5_path):
    with h5py.File(h5_path, "r") as f:
        return sorted(f["test"].keys())

### Connect your trained models to the evaluator

Implement the two adapter functions below to connect your trained models to the supplied evaluator. The deterministic adapter must call your trained predictor. The generative adapter must implement or call your flow-matching sampler, including Gaussian noise generation and numerical integration of the learned ODE; this sampling code is not supplied. The metric, boundary enforcement, autoregressive rollout loops, evaluation protocol, aggregation, and metric plotting are already implemented in the following code cells.

Each function must return the **full physical velocity field one second later**, with shape `(N, 2)`. If your model works with normalized velocities or normalized increments internally, perform the required denormalization inside the adapter before returning.

The evaluation code does not assume any particular neural-network architecture.

`generative_next_velocity` should produce a new conditional sample when the supplied random generator changes. After implementing both adapters, run the example calls provided at the ends of the evaluation and aggregation cells with your trained models to compute, summarize, and plot the results. Interpret the comparison using the metric definition and rollout visualizations.


In [ ]:
@torch.no_grad()
def deterministic_next_velocity(model, trajectory, current_velocity):
    """
    TODO: predict the full velocity field one physical second ahead.

    Parameters
    ----------
    model : trained model
        Your model, placed on DEVICE and set to evaluation mode before rollout.
    trajectory : dict[str, torch.Tensor]
        Static mesh information for one trajectory (no batch dimension):
        - mesh_pos: float32 tensor, shape (N, 2), containing node (x, y) coordinates.
        - cells: int64 tensor, shape (M, 3), containing triangle node indices.
        - node_type: int64 tensor, shape (N,), containing labels 0 (interior),
          4 (inflow), 5 (outflow), or 6 (wall).
        All tensors are on DEVICE. N is the number of nodes and M is the number
        of triangles; both can vary between trajectories. This dictionary does
        not contain velocity history or future ground-truth states.
    current_velocity : torch.Tensor
        Float32 tensor of shape (N, 2) on DEVICE, in physical, unnormalized
        units. Columns contain (u_x, u_y). This is the ground-truth initial
        state for the first step and the previous prediction for later steps.

    Returns
    -------
    torch.Tensor
        Full velocity field u(t + 1 s), shape (N, 2), on the same device and
        with the same dtype as current_velocity, in physical, unnormalized
        units. If the model predicts a normalized increment, denormalize it
        and add it to current_velocity before returning.

    Notes
    -----
    One call advances physical time by 1 second (100 stored data steps).
    Return values for all N nodes. The supplied rollout code restores the
    known inflow and wall velocities after each call; interior and outflow
    velocities remain predicted.
    Implement model-specific feature preparation, normalization, inference,
    and denormalization here. This adapter returns one deterministic prediction.
    """
    raise NotImplementedError


@torch.no_grad()
def generative_next_velocity(model, trajectory, current_velocity, generator):
    """
    TODO: sample a full velocity field one physical second ahead.

    Parameters
    ----------
    model : trained model
        Your model, placed on DEVICE and set to evaluation mode before rollout.
    trajectory : dict[str, torch.Tensor]
        Static mesh information for one trajectory (no batch dimension):
        - mesh_pos: float32 tensor, shape (N, 2), containing node (x, y) coordinates.
        - cells: int64 tensor, shape (M, 3), containing triangle node indices.
        - node_type: int64 tensor, shape (N,), containing labels 0 (interior),
          4 (inflow), 5 (outflow), or 6 (wall).
        All tensors are on DEVICE. N is the number of nodes and M is the number
        of triangles; both can vary between trajectories. This dictionary does
        not contain velocity history or future ground-truth states.
    current_velocity : torch.Tensor
        Float32 tensor of shape (N, 2) on DEVICE, in physical, unnormalized
        units. Columns contain (u_x, u_y). This is the ground-truth initial
        state for the first step and the previous prediction for later steps.
    generator : torch.Generator
        Random generator on DEVICE, seeded by the supplied rollout code.
        Use it for all sampling randomness, for example:
        torch.randn(current_velocity.shape, device=current_velocity.device,
                    dtype=current_velocity.dtype, generator=generator).
        Do not reset its seed inside this function. Its state advances across
        rollout steps, and separate rollouts receive distinct seeds.

    Returns
    -------
    torch.Tensor
        Full velocity field u(t + 1 s), shape (N, 2), on the same device and
        with the same dtype as current_velocity, in physical, unnormalized
        units. If the model predicts a normalized increment, denormalize it
        and add it to current_velocity before returning.

    Notes
    -----
    One call advances physical time by 1 second (100 stored data steps).
    Return values for all N nodes. The supplied rollout code restores the
    known inflow and wall velocities after each call; interior and outflow
    velocities remain predicted.
    Implement or call your conditional flow-matching sampler here: draw fresh
    Gaussian noise, integrate the learned vector field over flow time tau from
    0 to 1 conditioned on current_velocity and the mesh, and convert the sample
    to a physical velocity field. Flow time tau is separate from physical time.
    Noise generation and ODE integration are student tasks, not supplied by
    the evaluator. Return one sample per call, not an ensemble or sample mean.
    """
    raise NotImplementedError

In [ ]:
def enforce_known_boundaries(next_velocity, current_velocity, node_type):
    """Keep inflow/wall nodes fixed; interior and outflow nodes remain predicted."""
    out = next_velocity.clone()
    dynamic = dynamic_node_mask(node_type)
    out[~dynamic] = current_velocity[~dynamic]
    return out


@torch.no_grad()
def rollout_deterministic_generic(model, trajectory, n_steps=5):
    current = trajectory["velocity"][0].to(DEVICE)
    tr_static = {
        "mesh_pos": trajectory["mesh_pos"].to(DEVICE),
        "cells": trajectory["cells"].to(DEVICE),
        "node_type": trajectory["node_type"].to(DEVICE),
    }

    states = [current.detach().cpu()]
    for _ in range(n_steps):
        nxt = deterministic_next_velocity(model, tr_static, current)
        nxt = enforce_known_boundaries(nxt, current, tr_static["node_type"])
        current = nxt
        states.append(current.detach().cpu())

    return torch.stack(states)


@torch.no_grad()
def rollout_generative_generic(model, trajectory, n_steps=5, seed=0):
    current = trajectory["velocity"][0].to(DEVICE)
    tr_static = {
        "mesh_pos": trajectory["mesh_pos"].to(DEVICE),
        "cells": trajectory["cells"].to(DEVICE),
        "node_type": trajectory["node_type"].to(DEVICE),
    }

    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(int(seed))

    states = [current.detach().cpu()]
    for _ in range(n_steps):
        nxt = generative_next_velocity(model, tr_static, current, gen)
        nxt = enforce_known_boundaries(nxt, current, tr_static["node_type"])
        current = nxt
        states.append(current.detach().cpu())

    return torch.stack(states)

In [ ]:
# ============================================================
# Final rollout temporal-activity evaluation
# ============================================================

N_EVAL_TRAJ = 20
N_GENERATIVE_SAMPLES = 5

# Six 1-second model steps are needed to form the final scored
# interval 5 -> 6 s.
N_MODEL_ROLLOUT_STEPS = 6

# The first interval 0 -> 1 s is burn-in.
BURN_IN_STEPS = 1

EVAL_SEED = 1234


@torch.no_grad()
def evaluate_rollout_temporal_activity(
    det_model,
    generative_model,
):
    rows = []

    ids = test_trajectory_ids(DATA_PATH)[:N_EVAL_TRAJ]

    for traj_number, traj_id in enumerate(
        tqdm(
            ids,
            desc="Test trajectories",
        )
    ):
        tr = load_full_trajectory(
            DATA_PATH,
            "test",
            traj_id,
        )

        # ----------------------------------------------------
        # Generate model rollouts to nominal t = 6 s.
        # ----------------------------------------------------

        det = rollout_deterministic_generic(
            det_model,
            tr,
            n_steps=N_MODEL_ROLLOUT_STEPS,
        )

        fm_rollouts = []

        for sample_id in tqdm(
            range(N_GENERATIVE_SAMPLES),
            desc=f"Generative samples {traj_id}",
            leave=False,
        ):
            seed = (
                EVAL_SEED
                +
                1000 * traj_number
                +
                sample_id
            )

            fm_rollouts.append(
                rollout_generative_generic(
                    generative_model,
                    tr,
                    n_steps=N_MODEL_ROLLOUT_STEPS,
                    seed=seed,
                )
            )

        # ----------------------------------------------------
        # Score intervals:
        #
        #   1 -> 2
        #   2 -> 3
        #   3 -> 4
        #   4 -> 5
        #   5 -> 6 nominally
        #
        # 0 -> 1 is burn-in and is not scored.
        # ----------------------------------------------------

        T = tr["velocity"].shape[0]

        for end_step in range(
            BURN_IN_STEPS + 1,
            N_MODEL_ROLLOUT_STEPS + 1,
        ):
            start_step = end_step - 1

            nominal_start_s = (
                start_step
                *
                PREDICTION_DT
            )

            nominal_end_s = (
                end_step
                *
                PREDICTION_DT
            )

            # ------------------------------------------------
            # Ground-truth endpoints.
            #
            # For 1->2, ..., 4->5, exact 1-second endpoints
            # exist in the stored trajectory.
            #
            # For the final nominal 5->6 interval, use
            # 5.00 -> 5.99 s.
            # ------------------------------------------------

            gt_start_idx = (
                start_step
                *
                HORIZON_STEPS
            )

            desired_gt_end_idx = (
                end_step
                *
                HORIZON_STEPS
            )

            gt_end_idx = min(
                desired_gt_end_idx,
                T - 1,
            )

            gt_start_time_s = (
                gt_start_idx
                *
                DT
            )

            gt_end_time_s = (
                gt_end_idx
                *
                DT
            )

            gt_delta_t = (
                gt_end_time_s
                -
                gt_start_time_s
            )

            if gt_delta_t <= 0:
                continue

            gt_activity = (
                wake_velocity_temporal_activity(
                    tr["mesh_pos"],
                    tr["cells"],
                    tr["velocity"][gt_start_idx],
                    tr["velocity"][gt_end_idx],
                    gt_delta_t,
                )
            )

            rows.append({
                "trajectory": traj_id,
                "interval_start_s": nominal_start_s,
                "interval_end_s": nominal_end_s,
                "gt_interval_end_s": gt_end_time_s,
                "model": "Ground truth",
                "sample": -1,
                "temporal_activity": gt_activity,
            })

            # ------------------------------------------------
            # Deterministic model:
            # always a nominal 1-second model interval.
            # ------------------------------------------------

            det_activity = (
                wake_velocity_temporal_activity(
                    tr["mesh_pos"],
                    tr["cells"],
                    det[start_step],
                    det[end_step],
                    PREDICTION_DT,
                )
            )

            rows.append({
                "trajectory": traj_id,
                "interval_start_s": nominal_start_s,
                "interval_end_s": nominal_end_s,
                "gt_interval_end_s": gt_end_time_s,
                "model": "Deterministic",
                "sample": -1,
                "temporal_activity": det_activity,
            })

            # ------------------------------------------------
            # Flow matching: evaluate every sampled rollout.
            # ------------------------------------------------

            for sample_id, fm in enumerate(
                fm_rollouts
            ):
                fm_activity = (
                    wake_velocity_temporal_activity(
                        tr["mesh_pos"],
                        tr["cells"],
                        fm[start_step],
                        fm[end_step],
                        PREDICTION_DT,
                    )
                )

                rows.append({
                    "trajectory": traj_id,
                    "interval_start_s": nominal_start_s,
                    "interval_end_s": nominal_end_s,
                    "gt_interval_end_s": gt_end_time_s,
                    "model": "Flow matching",
                    "sample": sample_id,
                    "temporal_activity": fm_activity,
                })

    return pd.DataFrame(rows)


# Run after implementing the two model adapter functions above:
#
# rollout_activity_raw = evaluate_rollout_temporal_activity(
#     det_model,
#     flow_model,
# )


In [ ]:
# ============================================================
# Aggregate correctly and plot the metric
# ============================================================

def summarize_rollout_temporal_activity(
    rollout_activity_raw,
):
    # --------------------------------------------------------
    # First average independent FM samples within the same
    # trajectory and interval.
    #
    # GT and deterministic each already have one value.
    # --------------------------------------------------------

    per_condition = (
        rollout_activity_raw
        .groupby(
            [
                "model",
                "trajectory",
                "interval_start_s",
                "interval_end_s",
            ]
        )["temporal_activity"]
        .mean()
        .reset_index()
    )

    # --------------------------------------------------------
    # Then summarize across independent test trajectories.
    # --------------------------------------------------------

    by_time = (
        per_condition
        .groupby(
            [
                "model",
                "interval_start_s",
                "interval_end_s",
            ]
        )
        .agg(
            mean_activity=(
                "temporal_activity",
                "mean",
            ),
            std_activity=(
                "temporal_activity",
                "std",
            ),
            median_activity=(
                "temporal_activity",
                "median",
            ),
            n_trajectories=(
                "trajectory",
                "nunique",
            ),
        )
        .reset_index()
    )

    return (
        per_condition,
        by_time,
    )


def plot_rollout_temporal_activity(
    by_time,
    show_std=True,
):
    plt.figure(
        figsize=(8, 5)
    )

    for model_name in [
        "Ground truth",
        "Deterministic",
        "Flow matching",
    ]:
        d = (
            by_time[
                by_time["model"]
                ==
                model_name
            ]
            .sort_values(
                "interval_end_s"
            )
        )

        plt.plot(
            d["interval_end_s"],
            d["mean_activity"],
            marker="o",
            linewidth=2,
            label=model_name,
        )

        if show_std:
            plt.fill_between(
                d["interval_end_s"],
                d["mean_activity"]
                -
                d["std_activity"],
                d["mean_activity"]
                +
                d["std_activity"],
                alpha=0.16,
            )

    plt.xlabel(
        "End of interval [s]"
    )

    plt.ylabel(
        r"Wake temporal activity "
        r"$\langle \|\Delta\mathbf{u}\|/\Delta t\rangle_{\mathrm{wake}}$"
    )

    plt.title(
        "Absolute wake temporal activity during autoregressive rollout"
    )

    plt.xticks(
        [2, 3, 4, 5, 6]
    )

    plt.legend()
    plt.tight_layout()
    plt.show()


# Example after evaluation:
#
# per_condition, by_time = summarize_rollout_temporal_activity(
#     rollout_activity_raw
# )
#
# display(by_time)
# plot_rollout_temporal_activity(by_time)
